# TARGE — Prismatic VLM (Colab)

End-to-end pipeline: setup → install → login → paths → data → overview → train → eval.


## 1. Setup: GPU check + mount Drive


In [ ]:
print("== GPU ==")
!nvidia-smi -L || echo "(no GPU detected)"
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader || true

print("\n== Drive ==")
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")


## 2. Install dependencies


In [ ]:
import time, sys
%cd /content/drive/MyDrive/targe-prismatic-vlms
print("[install] pip install -e . (this can take ~1-2 min) ...")
t0 = time.time()
!pip install -e . --quiet
print(f"[install] done in {time.time()-t0:.1f}s")
# !pip install flash-attn --no-build-isolation --quiet  # optional speedup on A100/L4


## 3. Logins (Hugging Face, W&B)


In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login, whoami

# Hugging Face
hf_token = userdata.get('hf_token')
login(token=hf_token)
os.environ["HF_TOKEN"] = hf_token
who = whoami()
print(f"[hf] logged in as: {who.get('name')}  (orgs: {[o['name'] for o in who.get('orgs', [])]})")

# W&B (optional)
try:
    wandb_token = userdata.get('wandb_token')
    os.environ["WANDB_API_KEY"] = wandb_token
    print("[wandb] WANDB_API_KEY set from Colab secrets")
except Exception:
    print("[wandb] no wandb_token secret found — skipping (training will not log to W&B)")


## 4. Configure filepaths

All paths live here. Drive = persistent (slow); local = fast scratch on the Colab VM.


In [ ]:
import os
from pathlib import Path

# --- Drive (persistent) ---
DRIVE_ROOT     = Path("/content/drive/MyDrive/targe-prismatic-vlms")
DRIVE_DATA     = DRIVE_ROOT / "data"                 # source-of-truth dataset
DRIVE_RUNS     = DRIVE_ROOT / "runs"                 # checkpoints + logs (persisted)
DRIVE_HF_CACHE = DRIVE_ROOT / "hf_cache"             # HF model cache (persisted)

# --- Local (fast scratch on Colab VM) ---
LOCAL_ROOT = Path("/content/targe")
LOCAL_DATA = LOCAL_ROOT / "data"                     # data copied here for fast IO

# Dataset subpath (relative to DRIVE_DATA / LOCAL_DATA)
DATASET_SUBDIR = Path("download/llava-laion-cc-sbu-558k")
CHAT_JSON_NAME = "chat.pruned.json"                  # pruned to images that exist on disk

for p in (DRIVE_DATA, DRIVE_RUNS, DRIVE_HF_CACHE, LOCAL_ROOT, LOCAL_DATA):
    p.mkdir(parents=True, exist_ok=True)

# HF cache on Drive (model weights survive Colab session resets)
os.environ["HF_HOME"] = str(DRIVE_HF_CACHE)

# Repo lives on Drive, so runs/ already persists — no symlink needed.
REPO_ROOT = DRIVE_ROOT

print(f"REPO_ROOT  : {REPO_ROOT}")
print(f"DRIVE_DATA : {DRIVE_DATA}")
print(f"LOCAL_DATA : {LOCAL_DATA}")
print(f"DRIVE_RUNS : {DRIVE_RUNS}")
print(f"HF_HOME    : {os.environ['HF_HOME']}")


## 5. Download dataset subset from Hugging Face → local Colab disk

Pulls the first `N_ROWS` of `theblackcat102/llava-pretrain` directly to the Colab VM's SSD (no Drive round-trip). Saves with `save_to_disk` and reloads with `load_from_disk` for fast IO.


In [ ]:
import time
from pathlib import Path
from datasets import load_dataset, load_from_disk

HF_DATASET_ID = "theblackcat102/llava-pretrain"
N_ROWS        = 5_000
LOCAL_SUBSET  = LOCAL_ROOT / "llava_subset"   # arrow files live here

print(f"[hf] dataset : {HF_DATASET_ID}")
print(f"[hf] split   : train[:{N_ROWS}]")
print(f"[hf] target  : {LOCAL_SUBSET}")
print(f"[hf] cache   : {os.environ.get('HF_HOME', '<default>')}\n")

# 1. Download subset (skip if already saved)
if LOCAL_SUBSET.exists() and any(LOCAL_SUBSET.iterdir()):
    print(f"[skip] {LOCAL_SUBSET} already populated — re-using on-disk copy.")
else:
    print("[hf] downloading subset ...")
    t0 = time.time()
    subset = load_dataset(HF_DATASET_ID, split=f"train[:{N_ROWS}]")
    print(f"[hf] downloaded {len(subset):,} rows in {time.time()-t0:.1f}s")
    print(f"[hf] columns  : {subset.column_names}")
    print(f"[hf] features : {subset.features}\n")

    print(f"[save] writing arrow shards to {LOCAL_SUBSET} ...")
    t0 = time.time()
    LOCAL_SUBSET.parent.mkdir(parents=True, exist_ok=True)
    subset.save_to_disk(str(LOCAL_SUBSET))
    print(f"[save] done in {time.time()-t0:.1f}s")

# 2. Reload from local SSD — zero network, zero Drive latency
print("\n[load] reopening from local SSD ...")
t0 = time.time()
fast_local_dataset = load_from_disk(str(LOCAL_SUBSET))
print(f"[load] {fast_local_dataset}")
print(f"[load] reopened in {time.time()-t0:.2f}s")

# 3. Quick on-disk size
!du -sh {LOCAL_SUBSET}


## 6. Dataset overview


In [ ]:
import json
from collections import Counter

dataset_dir = LOCAL_DATA / DATASET_SUBDIR
chat_path   = dataset_dir / CHAT_JSON_NAME

# File counts
n_files = sum(1 for _ in dataset_dir.rglob("*") if _.is_file())
n_imgs  = sum(1 for p in dataset_dir.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})

# Chat-json stats
with open(chat_path) as f:
    chat = json.load(f)

n_entries     = len(chat)
n_with_image  = sum(1 for e in chat if e.get("image"))
n_text_only   = n_entries - n_with_image
turns_per_ex  = Counter(len(e.get("conversations", [])) for e in chat)

print(f"Dataset dir       : {dataset_dir}")
print(f"Total files       : {n_files:,}")
print(f"  image files     : {n_imgs:,}")
print(f"Chat JSON         : {chat_path.name}")
print(f"  total entries   : {n_entries:,}")
print(f"  with image      : {n_with_image:,}")
print(f"  text-only       : {n_text_only:,}")
print(f"  turns/example   : {dict(sorted(turns_per_ex.items()))}")

# This dataset has no formal train/val split — alignment uses all of it as train.
print("\nSplit             : train-only (no held-out val in llava-laion-cc-sbu-558k)")


### Inspect a sample by image ID

Pass any substring of an image filename (e.g. `"0074375"`) and the cell renders the image + its chat turns.


In [ ]:
import json
from pathlib import Path
from IPython.display import display, Markdown
from PIL import Image

# Load once
_chat_cache = {}
def _load_chat():
    if "chat" not in _chat_cache:
        with open(LOCAL_DATA / DATASET_SUBDIR / CHAT_JSON_NAME) as f:
            _chat_cache["chat"] = json.load(f)
        # index by image filename for fast lookup
        _chat_cache["by_img"] = {e.get("image", ""): e for e in _chat_cache["chat"] if e.get("image")}
    return _chat_cache["chat"], _chat_cache["by_img"]

def show_sample(image_id: str, max_thumb=512):
    """image_id can be a full path ('00237/002374375.jpg'), a filename, or any substring like '0074375'."""
    chat, by_img = _load_chat()
    matches = [k for k in by_img if image_id in k]
    if not matches:
        print(f"[miss] no chat entry whose image path contains {image_id!r}")
        return
    if len(matches) > 1:
        print(f"[note] {len(matches)} matches — showing first. Others: {matches[1:6]}{'...' if len(matches)>6 else ''}")
    key = matches[0]
    entry = by_img[key]
    img_path = LOCAL_DATA / DATASET_SUBDIR / key

    # Header
    display(Markdown(f"**id:** `{entry.get('id','?')}`  &nbsp;|&nbsp; **image:** `{key}`"))

    # Image
    if img_path.is_file():
        im = Image.open(img_path)
        w, h = im.size
        if max(w, h) > max_thumb:
            im.thumbnail((max_thumb, max_thumb))
        display(im)
        display(Markdown(f"<sub>{w}×{h}px · {img_path.stat().st_size/1024:.1f} KB · `{img_path}`</sub>"))
    else:
        print(f"[warn] image not found on disk: {img_path}")

    # Conversation
    md = ["**Conversation:**"]
    for turn in entry.get("conversations", []):
        role = turn.get("from", "?")
        text = turn.get("value", "").replace("<image>", "_<image>_")
        md.append(f"- **{role}:** {text}")
    display(Markdown("\n".join(md)))

# Example
show_sample("0074375")


## 7. Train (align stage)


In [ ]:
import os, time
os.environ["PYTHONUNBUFFERED"] = "1"

RUN_ID    = "targe-smollm2-next"
LOG_PATH  = DRIVE_RUNS / RUN_ID / "train.log"
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

# Pull tokens from this kernel and forward them inline on the torchrun command.
# os.environ assignments DO propagate to !-magic subshells in Colab, but rely-on-inheritance
# has bitten us before (empty-string token -> "Authorization: Bearer " -> illegal-header crash),
# so we prepend them explicitly per the recommended fix.
HF_TOKEN_VAL = os.environ.get("HF_TOKEN", "").strip()
WB_KEY_VAL   = os.environ.get("WANDB_API_KEY", "").strip()
assert HF_TOKEN_VAL, "HF_TOKEN is empty — re-run the logins cell (Colab Secrets -> hf_token)"
if not WB_KEY_VAL:
    print("[warn] WANDB_API_KEY is empty — wandb tracker will fail; remove 'wandb' from --trackers or set the secret.")

print(f"[train] run_id   : {RUN_ID}")
print(f"[train] log file : {LOG_PATH}  (also streams to this cell)")
print(f"[train] wandb    : nbusich-northeastern-university/targe")
print(f"[train] hf_token : <set, {len(HF_TOKEN_VAL)} chars>")
print(f"[train] wb_key   : <set, {len(WB_KEY_VAL)} chars>" if WB_KEY_VAL else "[train] wb_key   : <empty>")
print(f"[train] started  : {time.strftime('%Y-%m-%d %H:%M:%S')}\n")

%cd /content/drive/MyDrive/targe-prismatic-vlms
!set -o pipefail; HF_TOKEN="{HF_TOKEN_VAL}" HUGGING_FACE_HUB_TOKEN="{HF_TOKEN_VAL}" WANDB_API_KEY="{WB_KEY_VAL}" \
  stdbuf -oL -eL torchrun --standalone --nnodes 1 --nproc-per-node 1 scripts/pretrain.py \
    --model.type "targe-smollm2-360m-clipb-224px" \
    --stage align \
    --model.align_per_device_batch_size 14 \
    --model.align_global_batch_size 14 \
    --model.align_learning_rate 1e-4 \
    --model.align_max_steps 500 \
    --dataset.type "llava-v15" \
    --run_id "{RUN_ID}" \
    --trackers '[jsonl,wandb]' \
    --wandb_entity nbusich-northeastern-university \
    --hf_token HF_TOKEN \
    --wandb_project targe 2>&1 | tee -a "{LOG_PATH}"

print(f"\n[train] finished : {time.strftime('%Y-%m-%d %H:%M:%S')}")


## 8. Eval (interactive generation against a checkpoint)


In [ ]:
RUN_ID   = "targe-smollm2-next"
CKPT_DIR = DRIVE_RUNS / RUN_ID

ckpts = sorted((CKPT_DIR / "checkpoints").glob("*.pt"))
print(f"[eval] run dir   : {CKPT_DIR}")
print(f"[eval] checkpoints found ({len(ckpts)}):")
for c in ckpts:
    sz = c.stat().st_size / 1e9
    print(f"   - {c.name:40s}  {sz:6.2f} GB")

assert ckpts, f"No checkpoints found in {CKPT_DIR / 'checkpoints'}"
LATEST = ckpts[-1]
print(f"\n[eval] using latest: {LATEST.name}")


In [ ]:
%cd /content/drive/MyDrive/targe-prismatic-vlms
!python scripts/generate.py --model_path "{CKPT_DIR}"
